In [ ]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import os
import pandas as pd

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
import os
os.getcwd()

In [ ]:
os.listdir()

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

# Device (auto GPU / CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ResNet50
resnet_model = models.resnet50(weights=None)

# Replace final classifier for 4 classes
resnet_model.fc = nn.Linear(resnet_model.fc.in_features, 4)

# Load trained weights safely for CPU/GPU
state_dict = torch.load("FYP/resnet50_basic.pth", map_location=device)
resnet_model.load_state_dict(state_dict)

# Move model to device
resnet_model = resnet_model.to(device)

# Evaluation mode
resnet_model.eval()

In [ ]:
# EfficientNetB0
efficient_model = models.efficientnet_b0(pretrained=False)

# Freeze all conv layers
for param in efficient_model.features.parameters():
    param.requires_grad = False

# Replace classifier for 4 classes
efficient_model.classifier[1] = nn.Linear(efficient_model.classifier[1].in_features, 4)

# Load trained weights
efficient_model.load_state_dict(torch.load("FYP/efficientnetB0_basic.pth", map_location=device))

# Move to device and set eval
efficient_model.to(device)
efficient_model.eval()

In [ ]:
from torchvision import transforms

preprocess = transforms.Compose([
    transforms.Resize((224, 224)),   # match your model input size
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [ ]:
import os

test_root = "/content/FYP/Split/test"  # your test folder
class_names = ["512Glioma", "512Meningioma", "512Pituitary", "512Normal"]

# Make a list of all test images
test_images = []
for cls in class_names:
    cls_path = os.path.join(test_root, cls)
    for img_name in os.listdir(cls_path):
        test_images.append({
            "path": os.path.join(cls_path, img_name),
            "true_class": cls
        })

# check first 5 entries
test_images[:5]

Loops through all test images.

Converts each image to RGB.

Applies preprocessing (Resize, ToTensor, Normalize) and adds batch dimension.

Runs the image through EfficientNet without computing gradients.

Uses softmax to convert model outputs (logits) into probabilities.

Picks the class with the highest probability as the predicted class.

Stores the image ID, true class, and EfficientNet prediction in a list.

In [ ]:
resnet_results = []

for img_info in test_images:
    img = Image.open(img_info["path"]).convert("RGB")
    input_tensor = preprocess(img).unsqueeze(0).to(device)

    with torch.no_grad():
        resnet_probs = torch.softmax(resnet_model(input_tensor), dim=1)
        resnet_pred = resnet_probs.argmax(dim=1).item()

    resnet_results.append({
        "image_id": os.path.basename(img_info["path"]),
        "true_class": img_info["true_class"],
        "resnet_pred": class_names[resnet_pred]
    })

# Optional: check first 5 predictions
resnet_results[:5]

In [ ]:
efficient_results = []

for img_info in test_images:
    img = Image.open(img_info["path"]).convert("RGB")  # Convert to RGB
    input_tensor = preprocess(img).unsqueeze(0).to(device)  # Preprocess and batch dimension

    with torch.no_grad():
        effnet_probs = torch.softmax(efficient_model(input_tensor), dim=1)  # Softmax to get probabilities
        effnet_pred = effnet_probs.argmax(dim=1).item()  # Predicted class index

    efficient_results.append({
        "image_id": os.path.basename(img_info["path"]),
        "true_class": img_info["true_class"],
        "efficientnet_pred": class_names[effnet_pred]  # Map to readable class name
    })

# Optional: check first 5 predictions
efficient_results[:5]

Weights: Calculated as normalized validation accuracy of each model:(0.5 each)

In [ ]:
ensemble_results = []

for img_info in test_images:
    img = Image.open(img_info["path"]).convert("RGB")
    input_tensor = preprocess(img).unsqueeze(0).to(device)

    with torch.no_grad():
        # Get probabilities
        resnet_probs = torch.softmax(resnet_model(input_tensor), dim=1)
        effnet_probs = torch.softmax(efficient_model(input_tensor), dim=1)

        # Weighted soft voting (adjust weights if you want)
        ensemble_probs = 0.4 * resnet_probs + 0.6 * effnet_probs
        ensemble_pred = ensemble_probs.argmax(dim=1).item()

    ensemble_results.append({
        "image_id": os.path.basename(img_info["path"]),
        "true_class": img_info["true_class"],
        "ensemble_pred": class_names[ensemble_pred]
    })

# check first 5 ensemble predictions
ensemble_results[:5]

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# True labels and predicted labels
y_true = [r["true_class"] for r in ensemble_results]
y_pred = [r["ensemble_pred"] for r in ensemble_results]

# Compute confusion matrix
cm = confusion_matrix(y_true, y_pred, labels=class_names)

# Display with larger figure
fig, ax = plt.subplots(figsize=(12, 9))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(cmap="viridis", ax=ax, xticks_rotation=45)
plt.title("Ensemble Confusion Matrix")
plt.show()

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

# True labels and predicted labels (same as confusion matrix)
all_labels = [r["true_class"] for r in ensemble_results]
all_preds = [r["ensemble_pred"] for r in ensemble_results]

# Accuracy
test_acc = accuracy_score(all_labels, all_preds)
print(f"Ensemble Test Accuracy: {test_acc * 100:.2f}%")

# Detailed metrics
print(classification_report(all_labels, all_preds, target_names=class_names))

# Ensembling with a Confidence probability threshold

In [ ]:
threshold_conf = 0.6  # Confidence threshold
ensemble_results_conf = []  # store predictions for confidence threshold

for img_info in test_images:
    img = Image.open(img_info["path"]).convert("RGB")
    input_tensor = preprocess(img).unsqueeze(0).to(device)

    with torch.no_grad():
        # Get softmax probabilities from both models
        resnet_probs = torch.softmax(resnet_model(input_tensor), dim=1)
        effnet_probs = torch.softmax(efficient_model(input_tensor), dim=1)

        # Weighted soft voting
        ensemble_probs = 0.4 * resnet_probs + 0.6 * effnet_probs

        # Check maximum probability for confidence
        max_prob, pred_idx = torch.max(ensemble_probs, dim=1)

        if max_prob.item() >= threshold_conf:
            pred_class = class_names[pred_idx.item()]
        else:
            pred_class = "uncertain"

    ensemble_results_conf.append({
        "image_id": os.path.basename(img_info["path"]),
        "true_class": img_info["true_class"],
        "ensemble_pred": pred_class
    })

# Optional: check first 5 predictions
ensemble_results_conf[:5]

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

# True labels and predicted labels (confidence threshold ensemble)
all_labels_conf = [r["true_class"] for r in ensemble_results_conf]
all_preds_conf = [r["ensemble_pred"] for r in ensemble_results_conf]

# Accuracy
test_acc_conf = accuracy_score(all_labels_conf, all_preds_conf)
print(f"Confidence Threshold Ensemble Test Accuracy: {test_acc_conf * 100:.2f}%")

# Detailed metrics
print(classification_report(all_labels_conf, all_preds_conf, target_names=class_names + ["uncertain"]))

# Entropy threshold

In [ ]:
import torch
import torch.nn.functional as F
import math

# Store results with entropy
ensemble_results_entropy = []

for img_info in test_images:
    img = Image.open(img_info["path"]).convert("RGB")
    input_tensor = preprocess(img).unsqueeze(0).to(device)

    with torch.no_grad():
        # Get probabilities from both models
        resnet_probs = torch.softmax(resnet_model(input_tensor), dim=1)
        effnet_probs = torch.softmax(efficient_model(input_tensor), dim=1)

        # Weighted soft voting (same as before)
        ensemble_probs = 0.6 * resnet_probs + 0.4 * effnet_probs

        # Compute entropy: -Σ p * log(p)
        probs_np = ensemble_probs.cpu().numpy().flatten()
        entropy = -sum([p * math.log(p + 1e-10) for p in probs_np])  # add small epsilon to avoid log(0)

        # Prediction class (normal max)
        pred_idx = torch.argmax(ensemble_probs, dim=1).item()
        pred_class = class_names[pred_idx]

    # Store everything
    ensemble_results_entropy.append({
        "image_id": os.path.basename(img_info["path"]),
        "true_class": img_info["true_class"],
        "ensemble_pred": pred_class,
        "entropy": entropy,
        "probs": probs_np  # optional, store full probability vector if needed
    })

# Optional: check first 5 results
ensemble_results_entropy[:5]

## entropy selection

# Collect all entropy values
all_entropy = [r["entropy"] for r in ensemble_results_entropy]

# See statistics
import numpy as np
print("Min entropy:", np.min(all_entropy))
print("Max entropy:", np.max(all_entropy))
print("Median entropy:", np.median(all_entropy))
print("Mean entropy:", np.mean(all_entropy))

# Running the entropy threshold

In [ ]:
# Set your entropy threshold
entropy_threshold = 0.3# you can adjust this based on your dataset

# List to store filtered predictions
ensemble_results_entropy_thresh = []

for r in ensemble_results_entropy:
    if r["entropy"] <= entropy_threshold:
        # Low entropy → confident prediction
        pred_class = r["ensemble_pred"]
    else:
        # High entropy → uncertain
        pred_class = "uncertain"

    ensemble_results_entropy_thresh.append({
        "image_id": r["image_id"],
        "true_class": r["true_class"],
        "ensemble_pred": pred_class,
        "entropy": r["entropy"]
    })

# Optional: check first 5 results
ensemble_results_entropy_thresh[:5]

# Accuracy testing for entropy threshold

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

# True labels and predicted labels
all_labels_entropy = [r["true_class"] for r in ensemble_results_entropy_thresh]
all_preds_entropy = [r["ensemble_pred"] for r in ensemble_results_entropy_thresh]

# Accuracy
test_acc_entropy = accuracy_score(all_labels_entropy, all_preds_entropy)
print(f"Entropy Threshold Ensemble Test Accuracy: {test_acc_entropy * 100:.2f}%")

# Detailed metrics
print(classification_report(
    all_labels_entropy,
    all_preds_entropy,
    target_names=class_names + ["uncertain"]
))

In [ ]:
# Only keep predictions below threshold
accepted_mask = [r["entropy"] <= entropy_threshold for r in ensemble_results_entropy]
filtered_labels = [r["true_class"] for i,r in enumerate(ensemble_results_entropy) if accepted_mask[i]]
filtered_preds  = [r["ensemble_pred"] for i,r in enumerate(ensemble_results_entropy) if accepted_mask[i]]

filtered_acc = accuracy_score(filtered_labels, filtered_preds)
data_retention = len(filtered_labels)/len(ensemble_results_entropy) * 100

print(f"Filtered Accuracy: {filtered_acc*100:.2f}%")
print(f"Data Retention: {data_retention:.2f}%")